## Image segmentation.

We will practice more image processing. In many cases, images with continuous pixel intensity are binarized to contain only bindary values. For example, from grayscale to black-and-white.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from imageio import imread

img = imread("II_1_raw-117.tiff")

print(img.shape)

plt.figure(figsize=(10,5))
plt.imshow(img, cmap='gray')
plt.axis('off')
plt.show()

# I_norm = img/255

First, we will denoise the image. Use the filter you think work best to denoise the input image. Here, we try several filters.

In [ ]:
from skimage.restoration import denoise_nl_means, estimate_sigma

# img_f = I_norm.astype(float)

img_f = img.astype(float)
img_f = (img_f - img_f.min()) / (img_f.max() - img_f.min())

sigma_est = np.mean(estimate_sigma(img_f, channel_axis=None))

img_nlm = denoise_nl_means(
    img_f,
    h=0.8*sigma_est,
    patch_size=5,
    patch_distance=8,
    fast_mode=True,
    channel_axis=None)

In [ ]:
plt.imshow(img_nlm,cmap='gray')
plt.show()

In [ ]:
from skimage.restoration import denoise_bilateral

img_bilat = denoise_bilateral(
    img_f,
    sigma_color=0.05,
    sigma_spatial=3,
    channel_axis=None)

In [ ]:
plt.imshow(img_bilat,cmap='gray')
plt.show()

In [ ]:
from skimage.restoration import denoise_tv_chambolle

img_tv = denoise_tv_chambolle(
    img_f,
    weight=0.05)

In [ ]:
plt.imshow(img_tv,cmap='gray')
plt.show()

### Set up threshold to `mask` the intensity.

In [ ]:
from ipywidgets import interact, fixed
import matplotlib.pyplot as plt

def gray_threshold(im, vmin=0.0, vmax=1.0, s=True):

    b_img = (im > vmin) & (im < vmax)

    print("vmin =", vmin)
    print("vmax =", vmax)

    if s:
        plt.figure(figsize=(12,5))
    
        plt.subplot(1,2,1)
        plt.imshow(im, cmap='gray')
        plt.title("Original")
        plt.axis("off")
    
        plt.subplot(1,2,2)
        plt.imshow(b_img, cmap='gray')
        plt.title("Binary segmentation")
        plt.axis("off")
    
        plt.show()

    return b_img

In [ ]:
w = interact(
    gray_threshold,
    im=fixed(img_nlm),
    vmin=(0.0, 1.0, 0.01),
    vmax=(0.0, 1.0, 0.01),
    __manual=True
)

### Erosion and dilation 

&#9989; Do This -- Search what image dilation and erosion are. Give a brief description.

**Try different values of iterations to see the difference.**

In [ ]:
from scipy import misc, ndimage

img_clean = img_nlm
b_img = gray_threshold(img_clean, vmin=0.47, vmax=0.96, s=False)

after_dialation = ndimage.binary_erosion(b_img, iterations=5)

plt.imshow(after_dialation)
plt.show() 

In [ ]:
from scipy import misc, ndimage

b_img = gray_threshold(img_clean, vmin=0.47, vmax=0.96, s=False)

after_dialation = ndimage.binary_dilation(b_img, iterations=3)

plt.imshow(after_dialation)
plt.show() 

### Edge

This gives a 1-pixel-wide boundary. Erosion shrinks the object slightly. Subtracting: `original - eroded` leaves only the boundary.


In [ ]:
from scipy.ndimage import binary_erosion
import matplotlib.pyplot as plt

b_img = gray_threshold(img_clean, vmin=0.47, vmax=0.96)
edge = b_img ^ binary_erosion(b_img)

plt.imshow(edge, cmap='gray')
plt.show()

In [ ]:
from scipy.ndimage import binary_dilation, binary_erosion

edge = binary_dilation(b_img) ^ binary_erosion(b_img)
plt.imshow(edge, cmap='gray')
plt.show()

Sobel edge detector

Works on grayscale OR binary images. Produces gradient magnitude.

$$\nabla I = \bigg( \frac{\partial I}{\partial x}, \frac{\partial I}{\partial y}\bigg).$$

$$G_x = \begin{bmatrix}
-1 & 0 & 1 \\
-2 & 0 & 2 \\
-1 & 0 & 1
\end{bmatrix} ~~~~\text{and}~~~~G_y = \begin{bmatrix}
-1 & -2 & 1 \\
0 & 0 & 0 \\
1 & 2 & 1
\end{bmatrix}.$$

$$S_x = G_x * I ~~~\text{and}~~~~ S_y = G_y * I,$$

where '$*$' is convolution operation. 

- Continuous form:

$$(I* K) (x,y) = \int\int I(x-u,y-v)K(u,v) du dy,$$

- Discrete image form:

$$(I* K) [m,n] = \sum_i \sum_j I[m-i,n-j] K[i,j]$$.

$$\nabla I = \sqrt{S_x^2 + S_y^2}.$$

**Relation to finite differences**

Simple derivative:

$$\frac{\partial I}{\partial x} \approx I(x+1) - I(x-1).$$

In [ ]:
from scipy.ndimage import sobel
import numpy as np

sx = sobel(b_img.astype(float), axis=0)
sy = sobel(b_img.astype(float), axis=1)

edge = np.hypot(sx, sy)

plt.imshow(edge, cmap='gray')
plt.show()

### Canny edge detector

1. Smooth image (e.g., Gaussian)
2. Compute gradients (e.g., Sobel)
3. Thin edges
4. Keep only strong/connected edges (thresholding)

In [ ]:
from skimage.feature import canny

edge = canny(b_img.astype(float))

plt.imshow(edge, cmap='gray')
plt.show()

**Over plotting edges on the original image.**

In [ ]:
plt.figure(figsize=(8,8))

plt.imshow(b_img, cmap='gray')
plt.contour(edge, colors='red', linewidths=0.5)

plt.show()

&#9989; Do This --  Which edge detecting you think works the best?

### skeletonize 

`skeletonize` reduces a binary object to a thin 1-pixel-wide centerline while preserving its topology. Skeletonization repeatedly removes boundary pixels while:

- not breaking connectivity
- not deleting endpoints

until only the center structure remains.
It is often called:

- medial axis
- topological skeleton
- centerline extraction

Very useful for:

- crack networks
- pore networks
- fibers
- grain boundaries
- vascular structures

In [ ]:
from skimage.morphology import skeletonize

skel = skeletonize(b_img)

In [ ]:
# edge = canny(b_img.astype(float))
plt.figure(figsize=(8,8))
plt.imshow(skel, cmap='gray')
plt.show()

---
## Calculate useful properties

Images represent the geometries of materials microstructures. You can calculate microstructural properties from the image data. 

- Volume fraction.
- edge density.


In [ ]:
b_img = gray_threshold(img_nlm, vmin=0.47, vmax=0.96, s=False)

h, w = b_img.shape
solid_fraction = np.sum(b_img)/(h*w)

edge = b_img ^ binary_erosion(b_img)
edge_density = np.sum(edge)/(h*w)


print("solid_fraction:", solid_fraction, ";   ", "edge_density:",  edge_density)

### Particle counting.

We will use `ndimage.measurements.label` to count the numnber of connected pixel groups. 
- First, let's use erosion and dilation to break connections between adjacent particles.
- Then, use `ndimage.measurements.label`.

In [ ]:
from scipy import misc, ndimage

b_img = gray_threshold(img_clean, vmin=0.47, vmax=0.96, s=False)

after_erosion = ndimage.binary_erosion(b_img, iterations=6)

plt.imshow(after_erosion)
plt.show() 

In [ ]:
b_img = gray_threshold(img_clean, vmin=0.47, vmax=0.96, s=False)

after_dialation = ndimage.binary_dilation(after_erosion, iterations=5)

plt.imshow(after_dialation)
plt.show() 

In [ ]:
# %matplotlib inline

plt.figure(figsize=(20,10))
lab, num_features = ndimage.measurements.label(after_dialation)
plt.imshow(lab)
plt.colorbar()
print(num_features)

np.shape(lab)


### Image-based simulation

Below we use a technique, "Smoothed Boundary Method" with finite difference method to simulate the diffusion in the pore channels.

$$\frac{\partial C}{\partial t} = \frac{1}{\phi} \bigg(\nabla \phi \cdot D \nabla C \bigg).$$

Such that diffusion occurs only in the region $\phi > \epsilon.$

In [ ]:
from scipy.ndimage import gaussian_filter

b_int = after_dialation.astype(int)

b_small = b_int[:100,:100]
from skimage.transform import resize

img2 = resize(
    b_small,
    (201, 201),
    anti_aliasing=True)

img2 = gaussian_filter(img2, sigma=2)


sbm = (img2-np.min(img2))/(np.max(img2)-np.min(img2)) 
plt.imshow(sbm, cmap='gray')
plt.show()

sbm

In [ ]:
phi = 1- np.copy(sbm)

dx = 1
dt = 0.08
Dff = 1

Con = np.zeros((201,201))
Lap = np.copy(Con)

for it in range(1, 10001 + 1):

    # ∇ · (phi ∇Con)
    Lap[1:-1, 1:-1] = 0.5 / dx**2 * (
        (phi[2:, 1:-1] + phi[1:-1, 1:-1])* (Con[2:, 1:-1] - Con[1:-1, 1:-1])-
        (phi[1:-1, 1:-1] + phi[0:-2, 1:-1])* (Con[1:-1, 1:-1] - Con[0:-2, 1:-1])+
        (phi[1:-1, 2:] + phi[1:-1, 1:-1])* (Con[1:-1, 2:] - Con[1:-1, 1:-1])-
        (phi[1:-1, 1:-1] + phi[1:-1, 0:-2])* (Con[1:-1, 1:-1] - Con[1:-1, 0:-2]))

    # update only where phi > 1e-4
    mask = phi[1:-1, 1:-1] > 1.0e-4

    Con_inner = Con[1:-1, 1:-1]

    Con_inner[mask] = (Con_inner[mask]
        + dt * Dff * Lap[1:-1, 1:-1][mask] / phi[1:-1, 1:-1][mask])

    Con[1:-1, 1:-1] = Con_inner

    # Neumann boundary conditions
    Con[0, :]  = Con[1, :]
    Con[-1, :] = Con[-2, :]
    Con[:, 0]  = 1
    Con[:, -1] = 0



In [ ]:
plt.imshow(Con)
plt.show()

### 3D data

Following a similar concept, in 3D the microstructure is represented as a 3D voxel array. We will import a stacked-tiff file, which contains many images in one file. 

There are open to public battery graphite anode 3D data in 
`https://www.research-collection.ethz.ch/entities/researchdata/2f6d999d-9b22-40ab-ab8f-da0c43b34c82`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tifffile

# Load stacked TIFF
vol = tifffile.imread("II_1_bin.tif")

print(vol.shape)

binary_vol = vol.astype(bool)

In [ ]:
binary_small = binary_vol[0:50, 0:50, 0:50]

%matplotlib widget


fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(111, projection="3d")

ax.voxels(
    binary_small,
    edgecolor=None
)

ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.set_zlabel("Z")

plt.show()